# Semi-supervised learning: lab notebook

**Machine Learning (5CS4-05), Semi-supervised and reinforcement learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.neighbors import KNeighborsClassifier
from sklearn.semi_supervised import LabelPropagation, SelfTrainingClassifier
from sklearn.svm import SVC

### 1. Two moons, but only 2 labelled points per class

In [ ]:
# 1. Two moons, but only 2 labelled points per class
X, y = make_moons(n_samples=200, noise=0.08, random_state=0)
rng = np.random.default_rng(3)
labelled = np.concatenate([rng.choice(np.flatnonzero(y == c), 2, replace=False) for c in (0, 1)])
y_partial = -np.ones_like(y)                    # -1 means unlabelled
y_partial[labelled] = y[labelled]
unl = y_partial == -1
print("labelled points:", len(labelled), " unlabelled:", int(unl.sum()))

### 2. Supervised only: 1-NN on the labelled points

In [ ]:
# 2. Supervised only: 1-NN on the labelled points
sup = KNeighborsClassifier(1).fit(X[labelled], y[labelled])
print("supervised 1-NN accuracy on unlabelled:", round(sup.score(X[unl], y[unl]), 3))

### 3. Label propagation (graph based)

In [ ]:
# 3. Label propagation (graph based)
ls = LabelPropagation(kernel="rbf", gamma=20, max_iter=5000).fit(X, y_partial)
print("label propagation accuracy on unlabelled:", round((ls.transduction_[unl] == y[unl]).mean(), 3))

### 4. Label propagation by hand: F <- P F, clamp the labelled rows

In [ ]:
# 4. Label propagation by hand: F <- P F, clamp the labelled rows
sigma = 0.25
D2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)
W = np.exp(-D2 / (2 * sigma ** 2)); np.fill_diagonal(W, 0)
P = W / W.sum(axis=1, keepdims=True)
F = np.full(len(X), 0.5); F[labelled] = y[labelled]
for it in range(1, 301):
    F = P @ F; F[labelled] = y[labelled]
    if it in (1, 10, 50, 300):
        print(f"iteration {it:3d}: accuracy on unlabelled = {((F[unl] >= 0.5) == y[unl]).mean():.3f}")

### 5. Self-training with an SVM base classifier

In [ ]:
# 5. Self-training with an SVM base classifier
st = SelfTrainingClassifier(SVC(probability=True, gamma=2.0, random_state=0), threshold=0.8).fit(X, y_partial)
print("self-training accuracy on unlabelled:", round((st.predict(X[unl]) == y[unl]).mean(), 3),
      " pseudo-labelled points:", int((st.labeled_iter_ > 0).sum()))

## Expected output

Running all cells above should print:

```
labelled points: 4  unlabelled: 196
supervised 1-NN accuracy on unlabelled: 0.668
label propagation accuracy on unlabelled: 1.0
iteration   1: accuracy on unlabelled = 0.663
iteration  10: accuracy on unlabelled = 0.944
iteration  50: accuracy on unlabelled = 1.000
iteration 300: accuracy on unlabelled = 0.995
self-training accuracy on unlabelled: 0.791  pseudo-labelled points: 188
```

## Practice exercises

Try these in the cells below.

1. Reduce the labelled points to 1 per class on the moons data. Compare supervised 1-NN with LabelPropagation.
2. Raise the kernel width (gamma / sigma) and watch the accuracy. Why can a very wide kernel fail?
3. Run SelfTrainingClassifier with threshold 0.7 and 0.99 and compare how many points get pseudo-labels.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
